[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/reciprocal/reciprocal.ipynb)

# Lifts and Reciprocal Diagrams

Give every vertex of a planar triangle mesh a height and the mesh lifts to a polyhedral terrain. Read the terrain as an Airy stress function and its creases are the bars of a planar truss: how sharply the terrain bends across an edge is the force in that bar, tension where it bends convex and compression where it bends concave, and every interior vertex is in equilibrium by construction. The slopes of the terrain's faces are the vertices of a second mesh, the reciprocal diagram, with an edge across every bar, perpendicular to it and as long as the bar's force. Lifted onto the paraboloid, the reciprocal diagram is the Voronoi diagram and the forces are the cotangent weights.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import VGA3D as ga
from examples import instantiate
from examples.animation import save_figure, show_animation
from examples.surfaces.reciprocal import render

mesh_module = instantiate("examples.mesh", ga)
Mesh, mv = mesh_module.Mesh, mesh_module.mv
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()

# The grid: its vertices per side across the square from -1 to 1, how far its interior vertices are nudged,
# and the seed of the nudges.
side = 5
jitter = 0.08
grid_seed = 11
# How far each height breathes about the paraboloid, the seed of the phases, the frames of one breath and
# the duration of each.
amplitude = 0.02
phase_seed = 3
frames_count = 48
duration_ms = 80

across = np.linspace(-1, 1, side)
coords = np.stack(np.meshgrid(across, across, indexing="ij"), axis=-1).reshape(-1, 2)   # [vertices, 2]
inside = (np.abs(coords) < 1).all(axis=-1)
coords[inside] += jitter * np.random.default_rng(grid_seed).normal(size=(inside.sum(), 2))
mesh = Mesh._triangulate(mv(Vector.derive.within("x y"), coords).field())

## 1. Slopes, stresses and the reciprocal diagram

A face of the terrain is a plane, and a plane over the plane of the mesh is one vector, its slope and its height at the origin, whose inner product with a point lifted to unit height, `point + e3`, is the height there. Through the three corners at their heights it is the corners' heights times the reciprocal frame of the corners lifted to unit height: linear in the heights. Across an interior edge the slope jumps, and the jump is perpendicular to the edge, since both planes agree along it; over the edge's length it is the bar's stress. The faces' slopes are the vertices of the reciprocal diagram, and the jumps its edges. On the paraboloid, at half the squared distance from the centre, each face's slope is its circumcentre and each stress the cotangent weight that `Mesh.edge_ratio` holds; the bars across cocircular corners carry almost nothing.

In [ ]:
def slopes(mesh: Mesh, heights: np.ndarray) -> Vector:
    """The plane of each face of the terrain, through its corners at their heights, as the vector whose inner
    product with a point lifted to unit height is the height there: the corners' heights times the reciprocal
    frame of the corners lifted to unit height, linear in the heights, the heights on the last axis."""
    lifted = mesh.vertices.batch() + mv.z                                      # [vertices] Vector
    a, b, c = (lifted[mesh.faces[:, corner]] for corner in range(3))          # [faces] Vector each
    volume = (a ^ b ^ c).inverse()                                             # [faces] Trivector
    frame = stack([(b ^ c) * volume, (c ^ a) * volume, (a ^ b) * volume], axis=-1)   # [faces, 3] Vector
    return (frame * heights[..., mesh.faces]).sum(axis=-1)                     # [..., faces] Vector


def reciprocal(mesh: Mesh, heights: np.ndarray) -> Vector:
    """The vertices of the reciprocal diagram: the slope of each face, its height at the origin left out."""
    planes = slopes(mesh, heights)                                             # [..., faces] Vector
    return (planes ^ mv.z) * mv.z                                              # [..., faces] Vector


def beside(mesh: Mesh) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The interior edges, and the faces on their left and on their right, seen along the edge from its
    lower vertex to its higher."""
    edge = mesh.face_edges.reshape(-1)                                         # [faces * 3]
    face = np.repeat(np.arange(len(mesh.faces)), 3)                            # [faces * 3]
    along = mesh.face_edge_orientation.reshape(-1) > 0                        # [faces * 3]
    left, right = np.full(len(mesh.edges), -1), np.full(len(mesh.edges), -1)
    left[edge[along]], right[edge[~along]] = face[along], face[~along]
    inner = np.flatnonzero((left >= 0) & (right >= 0))
    return inner, left[inner], right[inner]


def stresses(mesh: Mesh, heights: np.ndarray) -> Scalar:
    """The stress of each interior edge: the jump in slope between the faces on either side, measured
    towards the corner across the edge, over the edge's length, positive in tension, where the terrain
    bends convex. Which face is called left does not matter: swapping them turns both the jump and the
    side the corner lies on."""
    inner, left, right = beside(mesh)                                          # [inner] each
    planes = slopes(mesh, heights)                                             # [..., faces] Vector
    jump = planes[..., left] - planes[..., right]                              # [..., inner] Vector
    bars = mesh.edge_vectors.batch()[inner]                                    # [inner] Vector
    # The corner across each edge in the face on its right, as its offset square to the bar.
    far = mesh.faces[right].sum(axis=-1) - mesh.edges[inner].sum(axis=-1)    # [inner]
    corner = mesh.vertices.batch()[far] - mesh.vertices.batch()[mesh.edges[inner, 0]]   # [inner] Vector
    across = (corner ^ bars) / bars                                            # [inner] Vector
    return -(jump | across) / (across.norm() * bars.norm())                    # [..., inner] Scalar


def folds(mesh: Mesh, heights: np.ndarray) -> Scalar:
    """How far the corner across each interior edge on its right lies above the plane of the face on its
    left: positive where the terrain bends convex."""
    inner, left, right = beside(mesh)                                          # [inner] each
    far = mesh.faces[right].sum(axis=-1) - mesh.edges[inner].sum(axis=-1)    # [inner]
    lifted = mesh.vertices.batch()[far] + mv.z                                 # [inner] Vector
    return heights[..., far] - (slopes(mesh, heights)[..., left] | lifted)     # [..., inner] Scalar


paraboloid = (coords**2).sum(axis=-1) / 2                                      # [vertices]
inner, left, right = beside(mesh)
lifted_stresses = stresses(mesh, paraboloid)                                   # [inner] Scalar
print(f"stresses from {render.scalars(lifted_stresses).min():.3f} to {render.scalars(lifted_stresses).max():.3f}; "
      f"largest difference from the cotangent weights {np.abs(render.scalars(lifted_stresses - mesh.edge_ratio.batch()[inner])).max():.1e}")
save_figure(render.draw(mesh, paraboloid, lifted_stresses, reciprocal(mesh, paraboloid)), "reciprocal");

## 2. Tension and compression

Now let every height breathe a little about the paraboloid, each with its own phase. Where the terrain still bends convex the bar stays in tension, blue; where a vertex rises far enough above its neighbours the terrain creases concave there, the bar goes into compression, red, and its reciprocal edge turns around, folding the reciprocal diagram over itself. Every interior vertex stays in equilibrium throughout, and every reciprocal edge stays across its bar: the heights only ever change how hard each bar pulls or pushes. The boundary vertices are where the truss is held, and the forces there are its loads.

In the language of graphic statics, the planar truss is the form diagram and the reciprocal diagram the force diagram: each closed polygon of the force diagram is the equilibrium of the joint it is drawn across, and the polyhedral terrain over the form diagram is its Maxwell lift.

In [ ]:
phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, len(paraboloid))
times = np.linspace(0, 2 * np.pi, frames_count, endpoint=False)[:, None]
breaths = paraboloid + amplitude * np.sin(times + phases)                      # [frames, vertices]
breath_stresses = stresses(mesh, breaths)                                      # [frames] Scalar[inner]
breath_duals = reciprocal(mesh, breaths)                                       # [frames, faces] Vector

compressed = (render.scalars(breath_stresses) < 0).sum(axis=-1)
print(f"bars in compression over the breath: {compressed.min()} to {compressed.max()} of {len(inner)}")
show_animation(render.animate(mesh, breaths, breath_stresses, breath_duals), "reciprocal_breath", duration_ms)

In [ ]:
# checks
# Every bar is in tension exactly where the terrain bends convex, and some bars go into compression.
assert np.array_equal(render.scalars(breath_stresses) > 0, render.scalars(folds(mesh, breaths)) > 0)
assert (render.scalars(breath_stresses) < 0).any()
# Every reciprocal edge is perpendicular to its bar, and every interior vertex is in equilibrium.
bars = mesh.edge_vectors.batch()[inner]                                        # [inner] Vector
assert np.abs(render.scalars((breath_duals[:, left] - breath_duals[:, right]) | bars)).max() < 1e-10
assert render.unbalanced(mesh, breath_stresses) < 1e-10
# On the paraboloid the stresses are the cotangent weights and the reciprocal vertices the circumcentres.
assert np.abs(render.scalars(lifted_stresses - mesh.edge_ratio.batch()[inner])).max() < 1e-10
centres = reciprocal(mesh, paraboloid)                                         # [faces] Vector
reach = [render.scalars((centres - corner.batch()) | (centres - corner.batch())) for corner in mesh.corners]
assert np.abs(reach[0] - reach[1]).max() < 1e-10 and np.abs(reach[0] - reach[2]).max() < 1e-10